# Main Steam Games Dataset: Data Audit

This notebook performs a read-only audit of the raw `games.csv` before any cleaning. It checks the file structure, column types, missing and blank values, AppID and row duplication, numeric ranges, release-date parseability, and possible text-encoding artifacts. The source CSV is not modified.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Find games.csv whether Jupyter starts in the project root or Data Prep.
candidates = [Path('games.csv'), Path('Data Prep/games.csv'), Path('../Data Prep/games.csv')]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find games.csv. Checked: ' + ', '.join(map(str, candidates)))

print(f'Working directory: {Path.cwd()}')
print(f'Dataset file: {data_path.resolve()}')
df = pd.read_csv(data_path, encoding='utf-8-sig', low_memory=False)
print(f'Rows: {len(df):,} | Columns: {len(df.columns):,}')
display(df.head(3))

## Schema and column types

Check that the delimiter fix produced separate `Discount` and `DLC count` columns, and review inferred data types.

In [ ]:
print('Column names:')
display(pd.DataFrame({'column': df.columns, 'dtype': df.dtypes.astype(str).values}))
print('Duplicate column names:', df.columns[df.columns.duplicated()].tolist())
print('Discount present:', 'Discount' in df.columns)
print('DLC count present:', 'DLC count' in df.columns)
print('Data types summary:')
display(df.dtypes.astype(str).value_counts().rename_axis('dtype').to_frame('column_count'))

## Missing values and blank strings

In [ ]:
missing_by_column = df.isna().sum().sort_values(ascending=False)
print(f'Total missing cells: {int(missing_by_column.sum()):,}')
display(missing_by_column[missing_by_column.gt(0)].rename('missing_cells').to_frame())
string_columns = df.select_dtypes(include=['object', 'string']).columns
blank_by_column = df[string_columns].apply(lambda col: col.astype('string').str.strip().eq('').sum()).sort_values(ascending=False)
print('Whitespace-only strings by column (not already parsed as missing):')
display(blank_by_column[blank_by_column.gt(0)].rename('blank_strings').to_frame())

## Duplicate records and AppID integrity

In [ ]:
print(f'Completely duplicated rows: {int(df.duplicated().sum()):,}')
if 'AppID' in df.columns:
    print(f'Missing AppIDs: {int(df["AppID"].isna().sum()):,}')
    print(f'Duplicate AppID rows after first occurrence: {int(df["AppID"].duplicated().sum()):,}')
    print(f'Unique AppIDs: {df["AppID"].nunique(dropna=True):,}')
    print('Sample repeated AppIDs:', df.loc[df['AppID'].duplicated(keep=False), 'AppID'].head(20).tolist())
if 'Name' in df.columns:
    print(f'Missing game names: {int(df["Name"].isna().sum()):,}')
    print(f'Duplicate nonmissing game names: {int(df["Name"].dropna().duplicated().sum()):,}')

## Numeric and boolean value checks

Summaries help flag impossible negatives, unexpected ranges, and columns that may need type conversion later. Zero values are counted separately because they may be valid.

In [ ]:
numeric_columns = df.select_dtypes(include='number').columns
numeric_summary = df[numeric_columns].describe().T
numeric_summary['zero_count'] = df[numeric_columns].eq(0).sum()
numeric_summary['negative_count'] = df[numeric_columns].lt(0).sum()
print(f'Numeric columns: {len(numeric_columns)}')
display(numeric_summary)
boolean_columns = df.select_dtypes(include='bool').columns
if len(boolean_columns):
    print('Boolean value counts:')
    display(pd.DataFrame({c: df[c].value_counts(dropna=False) for c in boolean_columns}).fillna(0).astype(int))

## Release-date and text checks

The release date is kept as supplied in the raw file; this check only measures parseability. Text scanning looks for common UTF-8 mojibake markers and replacement characters.

In [ ]:
if 'Release date' in df.columns:
    release_dates = pd.to_datetime(df['Release date'], errors='coerce')
    nonblank_dates = df['Release date'].notna() & df['Release date'].astype('string').str.strip().ne('')
    print(f'Nonblank release dates: {int(nonblank_dates.sum()):,}')
    print(f'Release dates not parsed by pandas: {int((nonblank_dates & release_dates.isna()).sum()):,}')
    print('Parsed release-date range:', release_dates.min(), 'to', release_dates.max())

markers = ['�', 'Ã', 'Â', 'â€', 'ðŸ']
text_findings = {}
for column in string_columns:
    text = df[column].dropna().astype(str)
    counts = {marker: int(text.str.contains(marker, regex=False).sum()) for marker in markers}
    if any(counts.values()):
        text_findings[column] = counts
print('Columns with common text-encoding warning markers:')
display(pd.DataFrame(text_findings).T if text_findings else pd.DataFrame(columns=markers))

## Audit notes

Missing values are not automatically errors: fields such as movies, URLs, reviews, and scores may be unavailable for many games. Use the audit results to choose cleaning rules before changing or exporting the data.

## Preparation step 1: remove unused fields, duplicates, empty descriptors/descriptions, and Test/Demo apps; append Notes and export

Keep the raw `df` unchanged. Remove rows identical in every source column except `AppID`, retaining the first row. Drop the agreed fields, append nonblank `Notes` to `About the game`, and remove the original `Notes` column. Remove rows where `Tags`, `Genres`, and `Categories` are all missing/blank, then review title-based whole-word `Playtest` and `Demo` candidates and exclude them. After appending Notes, count and remove rows with no description. Keep only `AppID`, `Name`, `About the game`, `Tags`, `Genres`, `Categories`, `Developers`, `Publishers`, and `Price` in the prepared dataframe. The optional display fields are retained for possible recommendation cards. Do not create a combined `Descriptors` column yet. Export the semi-processed dataframe as `games_semi_processed.csv` in the `Data Prep` folder. Description text is not yet normalized or translated.


In [ ]:
columns_to_drop = [
    'Discount', 'DLC count', 'Reviews', 'Website', 'Support url', 'Support email',
    'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative',
    'Score rank', 'Achievements', 'Recommendations',
    'Average playtime forever', 'Average playtime two weeks',
    'Median playtime forever', 'Median playtime two weeks',
    'Screenshots', 'Movies',
]
missing_columns = [column for column in columns_to_drop if column not in df.columns]
if missing_columns:
    raise KeyError(f'Expected columns to drop are missing: {missing_columns}')

# Drop duplicate records that match in every source column except AppID.
# Keep the first source row as the representative record.
duplicate_comparison_columns = [column for column in df.columns if column != 'AppID']
df_deduplicated = df.drop_duplicates(
    subset=duplicate_comparison_columns,
    keep='first',
).copy()
duplicate_content_rows_removed = len(df) - len(df_deduplicated)

df_clean = df_deduplicated.drop(columns=columns_to_drop).copy()
notes = df_clean['Notes']
descriptions = df_clean['About the game']

def append_notes_to_description(description, note):
    description = '' if pd.isna(description) else str(description)
    note = '' if pd.isna(note) else str(note)
    has_description = bool(description.strip())
    has_note = bool(note.strip())
    if has_description and has_note:
        return description.rstrip() + '\n\n' + note
    if has_description:
        return description
    if has_note:
        return note
    return pd.NA

df_clean['About the game'] = [
    append_notes_to_description(description, note)
    for description, note in zip(descriptions, notes)
]
df_clean = df_clean.drop(columns=['Notes'])

# Keep Tags, Genres, and Categories separate; remove rows where all three
# source fields are missing or whitespace-only.
descriptor_source_columns = ['Tags', 'Genres', 'Categories']
missing_descriptor_columns = [
    column for column in descriptor_source_columns if column not in df_clean.columns
]
if missing_descriptor_columns:
    raise KeyError(f'Descriptor source columns are missing: {missing_descriptor_columns}')
has_any_descriptor_source = df_clean[descriptor_source_columns].apply(
    lambda column: column.fillna('').astype('string').str.strip().ne('')
).any(axis=1)
rows_removed_for_empty_descriptors = int((~has_any_descriptor_source).sum())
df_clean = df_clean.loc[has_any_descriptor_source].copy()

# Review clearly labeled Playtest/Demo app titles before excluding them.
# Whole-word matching avoids false positives such as "Demon" or "Democracy".
name_text = df_clean['Name'].fillna('').astype('string')
is_playtest_title = name_text.str.contains(
    r'\bplay[\s_-]*test\b', case=False, regex=True, na=False
)
is_demo_title = name_text.str.contains(
    r'\bdemos?\b', case=False, regex=True, na=False
)
is_test_or_demo = is_playtest_title | is_demo_title
test_demo_review = df_clean.loc[
    is_test_or_demo,
    ['AppID', 'Name', 'Release date', 'About the game'],
].copy()
test_demo_review['Removal reason'] = [
    'Playtest and Demo' if playtest and demo else 'Playtest' if playtest else 'Demo'
    for playtest, demo in zip(
        is_playtest_title.loc[test_demo_review.index],
        is_demo_title.loc[test_demo_review.index],
    )
]
test_demo_review['Description preview'] = (
    test_demo_review['About the game'].fillna('').astype('string').str.slice(0, 400)
)
print(f'Confirmed title-based Test/Demo candidates for review: {len(test_demo_review):,}')
display(test_demo_review[['AppID', 'Name', 'Removal reason', 'Description preview']])
rows_removed_as_test_or_demo = int(is_test_or_demo.sum())
df_clean = df_clean.loc[~is_test_or_demo].copy()

# Notes have already been appended, so this is the final combined-description check.
has_combined_description = (
    df_clean['About the game'].fillna('').astype('string').str.strip().ne('')
)
rows_removed_for_empty_description = int((~has_combined_description).sum())
rows_before_description_filter = len(df_clean)
df_clean = df_clean.loc[has_combined_description].copy()

# Keep fields used for recommendations plus optional recommendation-card details.
columns_to_keep = [
    'AppID', 'Name', 'About the game', 'Tags', 'Genres', 'Categories',
    'Developers', 'Publishers', 'Price',
]
missing_kept_columns = [column for column in columns_to_keep if column not in df_clean.columns]
if missing_kept_columns:
    raise KeyError(f'Expected retained columns are missing: {missing_kept_columns}')
df_clean = df_clean[columns_to_keep].copy()

# Save the current semi-processed data; description normalization/translation comes later.
output_path = data_path.parent / 'games_semi_processed.csv'
df_clean.to_csv(output_path, index=False, encoding='utf-8-sig')

has_description = descriptions.fillna('').astype('string').str.strip().ne('')
has_notes = notes.fillna('').astype('string').str.strip().ne('')
print(f'Raw shape: {df.shape}')
print(f'Exact duplicate-content rows removed (excluding AppID): {duplicate_content_rows_removed:,}')
print(f'Rows removed because Tags, Genres, and Categories were all empty: {rows_removed_for_empty_descriptors:,}')
print(f'Playtest/Demo rows removed: {rows_removed_as_test_or_demo:,}')
print(f'Rows before empty-description filter: {rows_before_description_filter:,}')
print(f'Rows removed because description and Notes were both empty: {rows_removed_for_empty_description:,}')
print(f'Prepared in-memory shape: {df_clean.shape}')
print(f'Prepared columns: {df_clean.columns.tolist()}')
print(f'Dropped source columns: {len(df.columns) - len(df_clean.columns)}')
print(f'Notes appended to existing descriptions: {int((has_description & has_notes).sum()):,}')
print(f'Notes used where description was blank: {int((~has_description & has_notes).sum()):,}')
print(f'Rows with neither description nor notes before row filtering: {int((~has_description & ~has_notes).sum()):,}')
print('Notes column removed after appending:', 'Notes' not in df_clean.columns)
print('Descriptor source columns remain separate:', all(c in df_clean.columns for c in descriptor_source_columns))
print('Optional display fields retained:', all(c in df_clean.columns for c in ['Developers', 'Publishers', 'Price']))
print('Combined Descriptors column created:', 'Descriptors' in df_clean.columns)
print('CSV export performed: Yes')
print(f'Semi-processed CSV: {output_path.resolve()}')
preview_indices = [idx for idx in notes[has_notes].index if idx in df_clean.index][:3]
display(pd.DataFrame({
    'Name': df.loc[preview_indices, 'Name'],
    'Original description (preview)': descriptions.loc[preview_indices].astype('string').str.slice(0, 250),
    'Notes (preview)': notes.loc[preview_indices].astype('string').str.slice(0, 250),
    'Combined description (preview)': df_clean.loc[preview_indices, 'About the game'].astype('string').str.slice(0, 550),
}))